# Mass reconstruction and efficiency

SWAN notebook for the Upsilon early-science note. Unzip the entire package in your SWAN working directory, open this notebook, select the shared profile, then **Run All**. Keep the `upsilon_esr/` folder beside the notebooks.

The upper panel combines the three saved signal mass histograms and performs a **new simultaneous expected-signal DSCB fit**, with free state yields, fixed nominal mass differences, MC-calibrated tails/relative widths, and a common mass shift and width scale. No continuum background is supplied by these framework outputs. This is a conditional signal-only fit, not a validation of background-aware yield extraction.

The lower panels retain the saved finite-MC Wilson intervals. Their denominator is the eligible stored tuple rows; they must not be relabelled as a verified full-generation efficiency. The current state samples have different Q² and electron-ID settings.
The total fit uses a thicker red line beneath finer colored component dashes, so both remain visible where the curves coincide. The black expected-signal histogram is lighter; the fit method and table layout are retained; values are recomputed from the tagged inputs.

The top-panel table displays the fitted expected yields and DSCB Gaussian-core resolutions in MeV. The simplified efficiency-axis label does not change its tuple-conditional definition. Selection and model details remain in the audit, without a bottom text block on the figure.


**Repository profile:** copy `configs/plots_9x275.example.json` to `configs/plots.local.json` and edit that one file. Paths, runs, luminosity, selections, optional `display_rebin` and diagnostic permission are shared by all notebooks. See `docs/plotting.md`. Every execution creates a fresh output directory. The current figure layouts require all three states; the 1S-only 9×130 support run is reviewed through its status/audit outputs.


**Separate proton-tagged category.** The input reader requires the saved `jul26-reco-topology-v1` selection contract (topology 1/2 and finite positive tagged-proton momentum), applied to reconstruction only. Truth denominators and generated normalization remain unchanged. Run this notebook in a fresh kernel from this folder; do not mix modules or results with the original no-tag package.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from settings import load_profile, make_config, new_output_dir, apply_display_rebin, file_link_path
from upsilon_esr import preflight

# No ROOT or LaTeX installation is needed. If a Python dependency is missing,
# uncomment the next line, run it once, and restart the notebook kernel.
# %pip install -r requirements.txt


## Configuration

The example profile selects three separate proton-tagged full-framework runs at 2.5 fb⁻¹. No automatic search for the latest run is performed.

In [ ]:
# All paths, run IDs, selection metadata and diagnostic permission come from one profile.
# Default: ../configs/plots.local.json. Alternatively set UPSILON_PLOT_PROFILE
# before starting the SWAN kernel, or give load_profile("/path/to/profile.json").
PROFILE = load_profile()
OUTPUT_DIR = new_output_dir(PROFILE, "reconstruction")
cfg = make_config(profile=PROFILE, output_dir=OUTPUT_DIR)
print("Profile:", PROFILE["profile_path"])
print("Output directory:", OUTPUT_DIR)


## Validate inputs

This checks that the numerical tables exist and their state, luminosity and selection metadata agree. A mismatch is reported rather than silently relabelled.

In [ ]:
display(preflight(cfg, section="reconstruction"))

## Publication typography

The shared style is tuned for the single-column figures in the note. Source sizes are 16 pt for axis titles, 14.5 pt for ticks, 14 pt for legends, 13 pt for annotations, and 12.5–13.5 pt inside the slope inset. These become approximately 5.7–7.4 pt at the paper column width. The sizes are centralized in `upsilon_esr/common.py` as `FONT_SIZES`. Replacing the plotting modules requires a kernel restart before rerunning.


## Render the figure

In [ ]:
from upsilon_esr.reconstruction import plot_reconstruction

fig, audit = plot_reconstruction(
    cfg,
    rebin=2,  # 25 MeV upstream bins -> 50 MeV displayed bins; exact sum, no interpolation
    float_common_shape=True,
    efficiency_variables=("true_W", "true_pT"), # replace one with "true_y" if desired
    show_mass_intervals=False,  # line-only mass histogram; intervals remain in the CSV
)
plt.show()


## Numerical check

In [ ]:
display(pd.DataFrame(audit["states"]).T)
print("Expected-fit deviance:", audit["poisson_deviance"])
print("Common mass shift [GeV]:", audit["common_mass_offset_GeV"])
print("Common width scale:", audit["common_width_scale"])
for message in audit["warnings"]:
    print("Note:", message)


## Saved outputs

PDF is vector output for the paper; PNG is a quick preview. CSV/JSON companions record the displayed numbers and assumptions. Copy the three PDFs into the note’s `figures/proton_tagged/` directory.

In [ ]:
print("Output directory:", OUTPUT_DIR.resolve())
for suffix in ("pdf", "png"):
    display(FileLink(str(OUTPUT_DIR / ("upsilon_es_reconstruction." + suffix))))
display(FileLink(str(OUTPUT_DIR / "upsilon_es_reconstruction_audit.json")))